## Import libraries

In [1]:
import oci
import json
import base64
from dotenv import load_dotenv
import os
import ads
import pandas as pd

import oracledb


load_dotenv('/mnt/c/Git_Repos/oci-ai-playground/.env')
#load_dotenv('C:/Git_Repos/oci-ai-playground/.env')

#oracledb.defaults.config_dir = os.environ['TNS_ADMIN']   #we added this in .env file instead of hardcoding it here

ads.set_auth('api_key')


In [2]:
#a pycharm-WSL prevents jupyter notebooks from printing errors, so we need to do this:
import sys, traceback, IPython
def _show_tb(shell, etype, evalue, tb, tb_offset=None):
    traceback.print_exception(etype, evalue, tb, file=sys.stdout)
sys.stdout.flush()
get_ipython().set_custom_exc((Exception,), _show_tb)

In [3]:
#print(pd.DataFrame.ads.read_sql.__doc__)

## Configure connection and secret in vault for OML_USER

### Connect with OML_USER

In [4]:
df1=pd.DataFrame.ads.read_sql(
    f'''
    SELECT * FROM oml_user.CARDIOTOCOGRAPHY
    FETCH FIRST :rows_num ROWS ONLY
''',
    bind_variables={'rows_num': 10},
    connection_parameters={
    'user_name': 'OML_USER',
    'password': os.environ['OML_USER_PASSWORD'],
    'dsn':os.environ['OML_USER_DSN'],
    #'wallet_location': os.environ['ORACLETESTDB_WALLET_LOCATION'],
    #'wallet_password': os.environ['ORACLETESTDB_WALLET_PASSWORD']
}
)
df1

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,MAX,NMAX,NZEROS,MODE_VALUE,MEAN,MEDIAN,VARIANCE,TENDENCY,CLASS,NSP
0,120,0.000,0.0,0.000,0.000,0.0,0.000,73,0.5,43,...,126,2,0,120,137,121,73,1,9,2
1,132,0.006,0.0,0.006,0.003,0.0,0.000,17,2.1,0,...,198,6,1,141,136,140,12,0,6,1
2,133,0.003,0.0,0.008,0.003,0.0,0.000,16,2.1,0,...,198,5,1,141,135,138,13,0,6,1
3,134,0.003,0.0,0.008,0.003,0.0,0.000,16,2.4,0,...,170,11,0,137,134,137,13,1,6,1
4,132,0.007,0.0,0.008,0.000,0.0,0.000,16,2.4,0,...,170,9,0,137,136,138,11,1,2,1
5,134,0.001,0.0,0.010,0.009,0.0,0.002,26,5.9,0,...,200,5,3,76,107,107,170,0,8,3
6,134,0.001,0.0,0.013,0.008,0.0,0.003,29,6.3,0,...,200,6,3,71,107,106,215,0,8,3
7,122,0.000,0.0,0.000,0.000,0.0,0.000,83,0.5,6,...,130,0,0,122,122,123,3,1,9,3
8,122,0.000,0.0,0.002,0.000,0.0,0.000,84,0.5,5,...,130,0,0,122,122,123,3,1,9,3
9,122,0.000,0.0,0.003,0.000,0.0,0.000,86,0.3,6,...,130,1,0,122,122,123,1,1,9,3


## Create secret in vault with credentials for OML_USER

In [7]:
config = oci.config.from_file()

vault_client = oci.vault.VaultsClient(config)

vault_client.create_secret(
    create_secret_details=oci.vault.models.CreateSecretDetails(
        compartment_id=os.environ.get("COMPARTMENT_OCID"),
        secret_name="oml_user_creds",
        description="OML_USER credentials for OracleTestDB",
        vault_id=os.environ.get("MY_VAULT_OCID"),
        key_id=os.environ.get("MY_KEY_OCID"),
        secret_content=oci.vault.models.Base64SecretContentDetails(
            content_type="BASE64",
            content=base64.b64encode(
    json.dumps({
    "user_name": "OML_USER",
    "password": os.environ.get("OML_USER_PASSWORD"),
    "dsn": os.environ.get("OML_USER_DSN")
}).encode("utf-8")
).decode("utf-8")
        )
    )
)
print("Secret created successfully.")

Secret created successfully.


In [7]:
# Fetch secret from vault
config = oci.config.from_file()
secrets_client = oci.secrets.SecretsClient(config)

secret_bundle = secrets_client.get_secret_bundle(os.environ.get('OML_USER_CREDS_SECRET_OCID'))
secret_content = json.loads(
    base64.b64decode(
        secret_bundle.data.secret_bundle_content.content
    ).decode('utf-8')
)


df1 = pd.DataFrame.ads.read_sql(
    f'''
    SELECT * FROM oml_user.CARDIOTOCOGRAPHY
    FETCH FIRST :rows_num ROWS ONLY
    ''',
    bind_variables={'rows_num': 10},
    connection_parameters={
    'user_name': secret_content['user_name'],
    'password': secret_content['password'],
    'dsn': secret_content['dsn']
},
)
df1.head()

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,MAX,NMAX,NZEROS,MODE_VALUE,MEAN,MEDIAN,VARIANCE,TENDENCY,CLASS,NSP
0,120,0.000,0.0,0.000,0.000,0.0,0.0,73,0.5,43,...,126,2,0,120,137,121,73,1,9,2
1,132,0.006,0.0,0.006,0.003,0.0,0.0,17,2.1,0,...,198,6,1,141,136,140,12,0,6,1
2,133,0.003,0.0,0.008,0.003,0.0,0.0,16,2.1,0,...,198,5,1,141,135,138,13,0,6,1
3,134,0.003,0.0,0.008,0.003,0.0,0.0,16,2.4,0,...,170,11,0,137,134,137,13,1,6,1
4,132,0.007,0.0,0.008,0.000,0.0,0.0,16,2.4,0,...,170,9,0,137,136,138,11,1,2,1


## Same process for ADMIN user

In [4]:
df2=pd.DataFrame.ads.read_sql(
    f'''
    SELECT * FROM oml_user.CARDIOTOCOGRAPHY
    FETCH FIRST :rows_num ROWS ONLY
''',
    bind_variables={'rows_num': 10},
    connection_parameters={
    'user_name': 'ADMIN',
    'password': os.environ['ADMIN_PASSWORD'],
    'dsn':os.environ['ADMIN_DSN'],   #same as OML_USER
    #'wallet_location': os.environ['ORACLETESTDB_WALLET_LOCATION'],
    #'wallet_password': os.environ['ORACLETESTDB_WALLET_PASSWORD']
}
)
print(df2.shape)
df2.head()

(10, 23)


,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,MAX,NMAX,NZEROS,MODE_VALUE,MEAN,MEDIAN,VARIANCE,TENDENCY,CLASS,NSP
0,120,0.000,0.0,0.000,0.000,0.0,0.0,73,0.5,43,...,126,2,0,120,137,121,73,1,9,2
1,132,0.006,0.0,0.006,0.003,0.0,0.0,17,2.1,0,...,198,6,1,141,136,140,12,0,6,1
2,133,0.003,0.0,0.008,0.003,0.0,0.0,16,2.1,0,...,198,5,1,141,135,138,13,0,6,1
3,134,0.003,0.0,0.008,0.003,0.0,0.0,16,2.4,0,...,170,11,0,137,134,137,13,1,6,1
4,132,0.007,0.0,0.008,0.000,0.0,0.0,16,2.4,0,...,170,9,0,137,136,138,11,1,2,1


In [8]:
config = oci.config.from_file()

vault_client = oci.vault.VaultsClient(config)

vault_client.create_secret(
    create_secret_details=oci.vault.models.CreateSecretDetails(
        compartment_id=os.environ.get("COMPARTMENT_OCID"),
        secret_name="admin_user_creds",
        description="ADMIN credentials for OracleTestDB",
        vault_id=os.environ.get("MY_VAULT_OCID"),
        key_id=os.environ.get("MY_KEY_OCID"),
        secret_content=oci.vault.models.Base64SecretContentDetails(
            content_type="BASE64",
            content=base64.b64encode(
    json.dumps({
    "user_name": "ADMIN",
    "password": os.environ.get("ADMIN_PASSWORD"),
    "dsn": os.environ.get("ADMIN_DSN")
}).encode("utf-8")
).decode("utf-8")
        )
    )
)
print("Secret created successfully.")

Secret created successfully.


In [3]:
# Fetch secret from vault
config = oci.config.from_file()
secrets_client = oci.secrets.SecretsClient(config)

secret_bundle = secrets_client.get_secret_bundle(os.environ.get('ADMIN_SECRET_OCID'))
secret_content = json.loads(
    base64.b64decode(
        secret_bundle.data.secret_bundle_content.content
    ).decode('utf-8')
)


df1 = pd.DataFrame.ads.read_sql(
    f'''
    SELECT * FROM oml_user.CARDIOTOCOGRAPHY
    FETCH FIRST :rows_num ROWS ONLY
    ''',
    bind_variables={'rows_num': 10},
    connection_parameters={
    'user_name': secret_content['user_name'],
    'password': secret_content['password'],
    'dsn': secret_content['dsn']
},
)
df1.head()

,LB,AC,FM,UC,DL,DS,DP,ASTV,MSTV,ALTV,...,MAX,NMAX,NZEROS,MODE_VALUE,MEAN,MEDIAN,VARIANCE,TENDENCY,CLASS,NSP
0,120,0.000,0.0,0.000,0.000,0.0,0.0,73,0.5,43,...,126,2,0,120,137,121,73,1,9,2
1,132,0.006,0.0,0.006,0.003,0.0,0.0,17,2.1,0,...,198,6,1,141,136,140,12,0,6,1
2,133,0.003,0.0,0.008,0.003,0.0,0.0,16,2.1,0,...,198,5,1,141,135,138,13,0,6,1
3,134,0.003,0.0,0.008,0.003,0.0,0.0,16,2.4,0,...,170,11,0,137,134,137,13,1,6,1
4,132,0.007,0.0,0.008,0.000,0.0,0.0,16,2.4,0,...,170,9,0,137,136,138,11,1,2,1
